# Mitra Regressor — Use an Exported Predictor

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/mitra-regressor-pipeline)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/mitra-regressor-pipeline/blob/main/tutorials/mitra_regressor_predictor_inference_colab.ipynb)

This notebook uses the reusable `mitra-predictor.zip` exported by the main Mitra Regressor tutorial.

**You do not need the original DIMER ZIP, `model.safetensors`, `config.json`, or DIMER Workbench to use this exported predictor.**

Workflow:

`mitra-predictor.zip` → reload predictor → upload new CSV → predict → download `predictions.csv`


## 1. Install the matching runtime

The exported predictor was created with AutoGluon. This notebook pins the same Mitra-capable runtime used by the tutorial.

Pip may replace Colab's preinstalled PyTorch. If PyTorch was already imported and the installed version changes, restart the session and rerun from the top.


In [ ]:
import importlib.metadata as importlib_metadata
import sys

PREINSTALL_TORCH_VERSION = importlib_metadata.version('torch')
TORCH_WAS_IMPORTED = 'torch' in sys.modules
print('PyTorch before install:', PREINSTALL_TORCH_VERSION)

%pip install -q "autogluon.tabular[mitra]==1.5.0"

INSTALLED_TORCH_VERSION = importlib_metadata.version('torch')
AUTOGLUON_VERSION = importlib_metadata.version('autogluon.tabular')
if TORCH_WAS_IMPORTED and INSTALLED_TORCH_VERSION != PREINSTALL_TORCH_VERSION:
    raise RuntimeError('pip changed PyTorch after it had already been imported. Use Runtime → Restart session, then run the notebook top-to-bottom.')

import torch

print('AutoGluon:', AUTOGLUON_VERSION)
print('PyTorch:', torch.__version__)
print('PyTorch CUDA build:', torch.version.cuda)
print('CUDA available:', torch.cuda.is_available())


## 2. Upload and validate `mitra-predictor.zip`

Upload exactly one predictor ZIP exported by the main tutorial. The notebook computes its SHA-256, rejects unsafe archive paths/symlinks, extracts to a fresh directory, and locates the AutoGluon predictor root via `predictor.pkl`.

**Trust boundary:** `TabularPredictor.load(...)` deserializes Python model objects. Load only a predictor ZIP you exported yourself or received from a trusted source. Path-safe extraction does not make an untrusted predictor archive safe to deserialize. If you saved the SHA-256 printed by the export step, paste it into `EXPECTED_ZIP_SHA256` so this notebook can verify the archive before loading it.


In [ ]:
import hashlib
import shutil
import stat
import zipfile
from pathlib import Path

from google.colab import files

EXPECTED_ZIP_SHA256 = ''  # @param {type:'string'}
EXTRACT_ROOT = Path('/content/mitra-predictor-upload')
if EXTRACT_ROOT.exists():
    shutil.rmtree(EXTRACT_ROOT)
EXTRACT_ROOT.mkdir(parents=True)

uploaded = files.upload()
zips = [(name, payload) for name, payload in uploaded.items() if name.lower().endswith('.zip')]
if len(zips) != 1:
    raise RuntimeError('Upload exactly one mitra-predictor.zip file.')

zip_name, zip_payload = zips[0]
ZIP_PATH = Path('/content') / Path(zip_name).name
ZIP_PATH.write_bytes(zip_payload)

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()

def safe_extract_zip(zip_path, destination):
    destination = destination.resolve()
    with zipfile.ZipFile(zip_path) as z:
        for info in z.infolist():
            member = Path(info.filename)
            if member.is_absolute() or '..' in member.parts:
                raise RuntimeError(f'Unsafe archive member path: {info.filename!r}')
            mode = (info.external_attr >> 16) & 0o170000
            if mode == stat.S_IFLNK:
                raise RuntimeError(f'Symlink entries are not allowed: {info.filename!r}')
            target = (destination / member).resolve()
            if target != destination and destination not in target.parents:
                raise RuntimeError(f'Archive member escapes extraction root: {info.filename!r}')
        z.extractall(destination)

zip_digest = sha256_file(ZIP_PATH)
expected_zip_digest = EXPECTED_ZIP_SHA256.strip().lower()
if expected_zip_digest:
    if len(expected_zip_digest) != 64 or any(ch not in '0123456789abcdef' for ch in expected_zip_digest):
        raise ValueError('EXPECTED_ZIP_SHA256 must be a 64-character hexadecimal SHA-256 digest.')
    if zip_digest != expected_zip_digest:
        raise RuntimeError(f'Predictor ZIP checksum mismatch. Expected {expected_zip_digest}; got {zip_digest}.')
    print('✓ Predictor ZIP SHA-256 matches the expected digest.')
else:
    print('⚠ No expected predictor ZIP digest supplied; continue only if this archive came from a trusted source.')
print(f'ZIP: {ZIP_PATH.name}')
print(f'Size: {ZIP_PATH.stat().st_size / (1024 ** 2):.1f} MiB')
print('SHA-256:', zip_digest)

safe_extract_zip(ZIP_PATH, EXTRACT_ROOT)

candidates = sorted({p.parent for p in EXTRACT_ROOT.rglob('predictor.pkl')})
if len(candidates) != 1:
    raise RuntimeError(f'Expected exactly one AutoGluon predictor root containing predictor.pkl; found {len(candidates)}: {candidates}')

PREDICTOR_ROOT = candidates[0]
print('✓ Predictor root:', PREDICTOR_ROOT)


## 3. Load the predictor and inspect provenance

`tutorial_run_metadata.json` is read when present. The predictor itself is loaded with `TabularPredictor.load(...)`; individual `.pkl` and `.pt` files inside the archive are implementation details and should not be opened manually.


In [ ]:
import json
from pathlib import Path

import pandas as pd
from autogluon.tabular import TabularPredictor

METADATA_PATH = PREDICTOR_ROOT / 'tutorial_run_metadata.json'
run_metadata = {}
if METADATA_PATH.exists():
    run_metadata = json.loads(METADATA_PATH.read_text())
    recorded_ag = run_metadata.get('autogluon_version')
    if recorded_ag and recorded_ag != AUTOGLUON_VERSION:
        raise RuntimeError(
            f'Predictor was exported with AutoGluon {recorded_ag}, but this runtime has {AUTOGLUON_VERSION}. '
            'Use the recorded version for best compatibility.'
        )
else:
    print('⚠ tutorial_run_metadata.json not found; continuing with predictor-internal metadata.')

predictor = TabularPredictor.load(str(PREDICTOR_ROOT))
if predictor.problem_type != 'regression':
    raise RuntimeError(f'Expected a regression predictor, but loaded problem_type={predictor.problem_type!r}.')

if run_metadata.get('features'):
    FEATURE_COLUMNS = list(run_metadata['features'])
else:
    feature_metadata = getattr(predictor, 'feature_metadata_in', None)
    if feature_metadata is None:
        raise RuntimeError('Could not determine required input feature columns from predictor metadata.')
    FEATURE_COLUMNS = list(feature_metadata.get_features())

TARGET_COLUMN = run_metadata.get('target_column') or getattr(predictor, 'label', None)

summary = {
    'AutoGluon runtime': AUTOGLUON_VERSION,
    'Problem type': predictor.problem_type,
    'Evaluation metric': str(predictor.eval_metric),
    'Target column': TARGET_COLUMN,
    'Required features': len(FEATURE_COLUMNS),
    'Models': ', '.join(predictor.model_names()),
    'Export mode': run_metadata.get('mode', 'not recorded'),
    'Base model': run_metadata.get('base_model', 'not recorded'),
    'Base revision': run_metadata.get('base_model_revision', 'not recorded'),
}
display(pd.Series(summary, name='Predictor').to_frame())

print('Required feature columns:')
display(pd.DataFrame({'feature': FEATURE_COLUMNS}))

if run_metadata:
    provenance_keys = [
        'model_source',
        'weights_sha256',
        'config_sha256',
        'data_source',
        'sample_revision',
        'train_rows_used',
        'holdout_rows',
        'independent_test_rows',
        'exported_at_utc',
    ]
    provenance = {k: run_metadata.get(k) for k in provenance_keys if run_metadata.get(k) is not None}
    if provenance:
        display(pd.Series(provenance, name='Recorded value').to_frame())


## 4. Upload new rows for inference

Upload one CSV containing the same predictor columns used when the model was exported.

- Column order does not matter.
- Extra columns are preserved in the output but are not passed to the model.
- The target column is not required.
- Missing required predictor columns stop the run before inference.


In [ ]:
import csv
import io

def read_inference_csv(payload):
    text = payload.decode('utf-8-sig')
    rows = csv.reader(io.StringIO(text, newline=''))
    header = next((row for row in rows if row and not (len(row) == 1 and not row[0].strip())), [])
    seen = set()
    duplicates = []
    for name in header:
        if name in seen and name not in duplicates:
            duplicates.append(name)
        seen.add(name)
    if duplicates:
        raise ValueError(f'Inference CSV contains duplicate column names: {duplicates}')
    return pd.read_csv(io.BytesIO(payload))

uploaded = files.upload()
csvs = [(name, payload) for name, payload in uploaded.items() if name.lower().endswith('.csv')]
if len(csvs) != 1:
    raise RuntimeError('Upload exactly one inference CSV.')

csv_name, csv_payload = csvs[0]
new_data = read_inference_csv(csv_payload)

if new_data.columns.duplicated().any():
    duplicates = list(new_data.columns[new_data.columns.duplicated()])
    raise ValueError(f'Inference CSV contains duplicate column names: {duplicates}')

missing = [c for c in FEATURE_COLUMNS if c not in new_data.columns]
if missing:
    raise ValueError(f'Inference CSV is missing required feature columns: {missing}')

extra = [c for c in new_data.columns if c not in FEATURE_COLUMNS]
if extra:
    print(f'ℹ {len(extra)} extra column(s) will be preserved in predictions.csv but not used by the predictor: {extra}')

if 'prediction' in new_data.columns:
    raise ValueError("Inference CSV already contains a 'prediction' column; rename or remove it before running inference.")

X = new_data.reindex(columns=FEATURE_COLUMNS).copy()
print(f'✓ Ready for inference: {len(X):,} rows × {len(FEATURE_COLUMNS)} required features.')
display(X.head())


## 5. Predict and download `predictions.csv`

The output preserves the uploaded columns and adds one scalar `prediction` column.


In [ ]:
predictions = predictor.predict(X)

result = new_data.copy()
result['prediction'] = predictions.to_numpy()

OUTPUT_PATH = Path('/content/predictions.csv')
result.to_csv(OUTPUT_PATH, index=False)

display(result.head())
print(f'✓ Wrote {len(result):,} predictions to {OUTPUT_PATH}')
files.download(str(OUTPUT_PATH))


## AI use and provenance

This inference tutorial was developed with substantial AI assistance using **GPT-5.6 Sol High** under human direction and review.

- AI model/configuration: **GPT-5.6 Sol High**
- Provider/client: **OpenAI / ChatGPT**
- Agent Relay role: **Builder**
- Base-model developer: **AutoGluon team, Amazon Web Services (AWS)**
- Predictor provenance: read from `tutorial_run_metadata.json` when available

AI attribution is **provenance, not sign-off** and does not independently verify correctness.
